[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/Rang/blob/main/examples/rang_r_colab.ipynb)

# Rang examples in R

This notebook shows how to use Rang for categorical and continuous data with ggplot2. It installs Rang from GitHub and downloads a public dataset, so it can run in a fresh Google Colab session.

The notebook declares an R runtime. If Colab asks you to choose a runtime, select **R**, then choose **Runtime > Run all**.

## 1. Install Rang

The package lives in the `r` folder of the Rang repository. The setup cell installs any missing helpers and then installs the current package directly from GitHub.

In [ ]:
options(repos = c(CRAN = "https://cloud.r-project.org"))
needed <- c("remotes", "ggplot2")
missing <- needed[!vapply(needed, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing) > 0) {
  install.packages(missing)
}

remotes::install_github(
  "mohsennasab/Rang",
  subdir = "r",
  dependencies = FALSE,
  upgrade = "never",
  quiet = TRUE
)

In [ ]:
library(Rang)
library(ggplot2)

names(rang_palettes)

## 2. Load public online data

We will use the [Palmer Penguins dataset](https://allisonhorst.github.io/palmerpenguins/). It contains measurements for three penguin species observed in the Palmer Archipelago, Antarctica.

In [ ]:
data_url <- "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
penguins <- read.csv(data_url, na.strings = c("", "NA"))
cat("Loaded", nrow(penguins), "rows from the online dataset\n")
head(penguins)

## 3. View the palette ramps

Calling `rang(name)` returns the complete stored ramp. The order shown here is the order used for continuous interpolation.

In [ ]:
palette_names <- names(rang_palettes)
old_par <- par(mfrow = c(length(palette_names), 1), mar = c(0.4, 1, 2.2, 1))

for (name in palette_names) {
  colors <- as.character(rang(name))
  barplot(
    rep(1, length(colors)),
    col = colors,
    border = NA,
    space = 0,
    axes = FALSE,
    main = name
  )
}

invisible(par(old_par))

## 4. Use a discrete palette for categories

`scale_fill_rang_d()` fills a categorical variable straight from a Rang palette. It follows the stored pick order, so a small number of categories stays well separated. Here Golestan distinguishes the three penguin species.

In [ ]:
species_mass <- aggregate(body_mass_g ~ species, data = penguins, FUN = mean, na.rm = TRUE)

ggplot(species_mass, aes(x = species, y = body_mass_g, fill = species)) +
  geom_col(color = "white", linewidth = 0.4) +
  scale_fill_rang_d("Golestan", guide = "none") +
  labs(
    title = "Average penguin body mass",
    x = "Species",
    y = "Body mass, g"
  ) +
  theme_minimal(base_size = 12)

## 5. Use a continuous ramp for measurements

`scale_color_rang_c()` maps a numeric variable onto a smooth Rang ramp. Termeh runs steadily from light to dark, which is what ordered measurements such as elevation, depth and the body-mass values below need.

In [ ]:
complete_rows <- complete.cases(
  penguins[, c("bill_length_mm", "flipper_length_mm", "body_mass_g")]
)
plot_data <- penguins[complete_rows, ]

ggplot(
  plot_data,
  aes(x = bill_length_mm, y = flipper_length_mm, color = body_mass_g)
) +
  geom_point(size = 2.6, alpha = 0.9) +
  scale_color_rang_c("Termeh", name = "Body mass, g") +
  labs(
    title = "Penguin measurements with the Termeh ramp",
    x = "Bill length, mm",
    y = "Flipper length, mm"
  ) +
  theme_minimal(base_size = 12)

## 6. Inspect the artwork source

Every palette carries provenance metadata. This makes it possible to keep the visual result connected to the artwork that inspired it.

In [ ]:
rang_palettes[["Termeh"]]$source

## Next steps

Try replacing `Golestan` or `Termeh` with another name from `names(rang_palettes)`. Use `direction = -1` with any scale when the visual scale should run in the opposite direction.

Every scale comes in four spellings. Use `scale_fill_*` or `scale_color_*` to match the aesthetic, and `_d` for categories against `_c` for numbers. `scale_colour_*` works as an alias.

Palette data are available under CC0 1.0. The Rang R software is MIT licensed.